In [1]:
import torch 
import torch.nn as nn

In [2]:
input = torch.tensor([
    [0.55, 0.60 , 0.89],
    [0.57, 0.85, 0.64],
    [0.15, 0.80, 0.45],
    [0.78, 0.25, 0.10],
])


In [3]:
query = input[2]
attn_score = torch.empty(input.shape[0])
print(attn_score)
for i, x_i in enumerate(input):
    attn_score[i] = torch.dot(x_i,query)
print(attn_score)
print(x_i)


tensor([0., 0., 0., 0.])
tensor([0.9630, 1.0535, 0.8650, 0.3620])
tensor([0.7800, 0.2500, 0.1000])


In [4]:
res = 0
for i,element in enumerate(input[0]):
    res += input[0][i] * query[i]
print(res)

tensor(0.9630)


In [5]:
attn_weight = attn_score / attn_score.sum()
print("Attention weights:", attn_weight)
print("Sum:", attn_weight.sum())

Attention weights: tensor([0.2969, 0.3248, 0.2667, 0.1116])
Sum: tensor(1.)


In [7]:
def softmax(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)
print(attn_score)
attn_softmax_weights = softmax(attn_score)
print(attn_softmax_weights)
print(attn_softmax_weights.sum())

tensor([0.9630, 1.0535, 0.8650, 0.3620])
tensor([0.2817, 0.3084, 0.2554, 0.1545])
tensor(1.)


In [8]:
attn_weight_2 = torch.softmax(attn_score,  dim = 0)
print(attn_weight_2)

tensor([0.2817, 0.3084, 0.2554, 0.1545])


In [9]:

query = input[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(input):
    context_vec_2 += attn_weight_2[i]*x_i

print(context_vec_2)

tensor([0.4895, 0.6741, 0.5785])


In [10]:
arr = torch.empty(4,4)

# array = (
#     [[2 , 3, 4],
#     [2 ,4 ,7],
#     [2, 5, 6],
#     [3, 4 ,5]]
# )

for i, x_i in enumerate(input):
    for j , x_j in enumerate(input):
        arr[i, j] = torch.dot(x_i, x_j)

print(arr)

tensor([[1.4546, 1.3931, 0.9630, 0.6680],
        [1.3931, 1.4570, 1.0535, 0.7211],
        [0.9630, 1.0535, 0.8650, 0.3620],
        [0.6680, 0.7211, 0.3620, 0.6809]])


In [11]:
# @ matrix multiplication operator

transpose_input = input @ input.T
print(transpose_input)

tensor([[1.4546, 1.3931, 0.9630, 0.6680],
        [1.3931, 1.4570, 1.0535, 0.7211],
        [0.9630, 1.0535, 0.8650, 0.3620],
        [0.6680, 0.7211, 0.3620, 0.6809]])


In [12]:
#dim = 0 for rows iteration(dim = -2)
# dim = 1 for column iteration (dim = -1)

transpose_input_weight = torch.softmax(transpose_input, dim = 1)
print(transpose_input_weight)

tensor([[0.3325, 0.3127, 0.2034, 0.1514],
        [0.3041, 0.3241, 0.2165, 0.1553],
        [0.2817, 0.3084, 0.2554, 0.1545],
        [0.2629, 0.2772, 0.1936, 0.2663]])


In [13]:
x_2 = input[1]
d_in = input.shape[1]
d_out =  2

In [14]:
print(torch.manual_seed(15))

w_q = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_k = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_v = torch.nn.Parameter(torch.rand(d_in,  d_out),requires_grad=False)
print(w_q)

Parameter containing:
tensor([[0.2973, 0.2766],
        [0.7974, 0.3869],
        [0.9170, 0.4125]])


In [15]:
print(input)
query = x_2 @ w_q
key = x_2 @ w_k
value = x_2 @ w_v
print(x_2 , ":" , w_q , ":", w_k, ":", w_v)
print(query, ":" , key , ":" , value)

tensor([[0.5500, 0.6000, 0.8900],
        [0.5700, 0.8500, 0.6400],
        [0.1500, 0.8000, 0.4500],
        [0.7800, 0.2500, 0.1000]])
tensor([0.5700, 0.8500, 0.6400]) : Parameter containing:
tensor([[0.2973, 0.2766],
        [0.7974, 0.3869],
        [0.9170, 0.4125]]) : Parameter containing:
tensor([[0.5538, 0.5646],
        [0.5026, 0.2206],
        [0.6801, 0.3880]]) : Parameter containing:
tensor([[0.3152, 0.5083],
        [0.9454, 0.3008],
        [0.6058, 0.2999]])
tensor([1.4341, 0.7505]) : tensor([1.1782, 0.7576]) : tensor([1.3710, 0.7374])


In [16]:
keys = input @ w_k
values = input @ w_v
print(keys.shape)
print(values)

torch.Size([4, 2])
tensor([[1.2798, 0.7269],
        [1.3710, 0.7374],
        [1.0762, 0.4518],
        [0.5428, 0.5017]])


In [17]:
keys_2 = keys[1]
k_v_attn_score = query.dot(keys_2)

#dot product  of query and keys_2
for i in range(len(keys_2)):
    print(keys_2[i] * query[i])
print(k_v_attn_score)


tensor(1.6896)
tensor(0.5686)
tensor(2.2582)


In [18]:
q_k_t = query @ keys.T
print(q_k_t)

tensor([2.3289, 2.2582, 1.4617, 1.2983])


In [19]:
d_k = keys.shape[1]
softmax_weights = torch.softmax(q_k_t / d_k**0.5, dim=-1)
print(softmax_weights)

tensor([0.3361, 0.3197, 0.1820, 0.1622])


In [20]:
class Self_Attention_v1(nn.Module):


    def __init__(self, din, dout):
        super().__init__()
        self.w_query = nn.Parameter(torch.rand(din, dout))
        self.w_key = nn.Parameter(torch.rand(din, dout))
        self.w_value = nn.Parameter(torch.rand(din, dout))
    
    def forward(self, x):
        query = x @ self.w_query
        key = x @ self.w_key
        value = x @ self.w_value

        attn_score = query @ key.T
        attn_weights = torch.softmax(
          attn_score / key.shape[-1]**0.5, dim=-1
       )
        context_vec = attn_softmax_weights @ values
        return context_vec


torch.manual_seed(123)
obj1 = Self_Attention_v1(d_in,d_out)
print(obj1(input))

# class SelfAttention_v1(nn.Module):

#     def __init__(self, d_in, d_out):
#         super().__init__()
#         self.W_query = nn.Parameter(torch.rand(d_in, d_out))
#         self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
#         self.W_value = nn.Parameter(torch.rand(d_in, d_out))

#     def forward(self, x):
#         keys = x @ self.W_key
#         queries = x @ self.W_query
#         values = x @ self.W_value
        
#         attn_scores = queries @ keys.T # omega
#         attn_weights = torch.softmax(
#             attn_scores / keys.shape[-1]**0.5, dim=-1
#         )

#         context_vec = attn_weights @ values
#         return context_vec

# torch.manual_seed(123)
# sa_v1 = SelfAttention_v1(d_in, d_out)
# print(sa_v1(input))

tensor([1.1421, 0.6251])


In [23]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.v2_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.v2_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.v2_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self,x):
        keys = self.v2_key(x)
        queries = self.v2_query(x)
        values = self.v2_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(input))

tensor([[-0.0586,  0.1121],
        [-0.0598,  0.1106],
        [-0.0604,  0.1091],
        [-0.0607,  0.1081]], grad_fn=<MmBackward0>)


In [26]:
q = sa_v2.v2_query(input)
k = sa_v2.v2_key(input)
attn_score_v2 = q @ k.T


v2_attn_w = torch.softmax(attn_score_v2 / k.shape[-1]**0.5, dim=-1)
print(v2_attn_w)

tensor([[0.2806, 0.2475, 0.2178, 0.2541],
        [0.2827, 0.2488, 0.2189, 0.2495],
        [0.2740, 0.2497, 0.2277, 0.2486],
        [0.2655, 0.2501, 0.2358, 0.2486]], grad_fn=<SoftmaxBackward0>)


In [28]:
context_length = attn_score_v2.shape[0]
mask = torch.tril(torch.ones(context_length, context_length))
print(mask)
print(context_length)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])
4


In [32]:
mask_attn_w = mask*v2_attn_w
row_sums = mask_attn_w.sum(dim=0, keepdim=True)
masked_simple_norm = mask_attn_w / row_sums
print(masked_simple_norm)

tensor([[0.2545, 0.0000, 0.0000, 0.0000],
        [0.2564, 0.3324, 0.0000, 0.0000],
        [0.2484, 0.3336, 0.4914, 0.0000],
        [0.2408, 0.3341, 0.5086, 1.0000]], grad_fn=<DivBackward0>)


In [34]:
#Random Masking

torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [37]:

torch.manual_seed(123)
print(dropout(v2_attn_w))

tensor([[0.5613, 0.4950, 0.0000, 0.5081],
        [0.5654, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4994, 0.0000, 0.4972],
        [0.5310, 0.5002, 0.4715, 0.4972]], grad_fn=<MulBackward0>)


In [39]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.c_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.c_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.c_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.c_key(x)
        queries = self.c_query(x)
        values = self.c_value(x)

        c_attn_score = queries @ keys.transpose(1,2)
        c_attn_score.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            c_attn_score / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

batch = torch.stack((input, input), dim=0)
print(batch.shape)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 4, 3])
tensor([[[-0.6684,  0.0098],
         [-0.6906, -0.0941],
         [-0.6104, -0.1293],
         [-0.5831, -0.1357]],

        [[-0.6684,  0.0098],
         [-0.6906, -0.0941],
         [-0.6104, -0.1293],
         [-0.5831, -0.1357]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 4, 2])
